# Raw data grab: one channel per notebook, many notebooks at once

Downloads one channel of upstream data for every fire in a Hugging Face dataset repo. The fires are the repo's own
`catalog/fires.csv`, so run `kaggle_catalog.ipynb` on the same `HF_REPO` first. Each fire ends up with one folder:

```
fires/<fire_id>/
  LINKS.md, links.json    every channel of this fire: status, links to its files here, and the upstream URLs
  history/  perimeters/  firms/  gridmet/  ics209/  suppression/  tessera/  terrain/
  landfire/  lulc/  landcover/  vegetation/  goes/  hrrr/  hrrr_next/  gfs/  osm/  viirs/
```

**One copy per channel.** Set `CHANNEL` below, and make one copy of this notebook for each channel you want running.
All copies upload to the same `HF_REPO` at the same time. This is safe for two reasons:
- each channel writes only its own files (`fires/<id>/<channel>/` and `manifest/<channel>/`);
- an upload that meets a busy Hub is retried with growing waits.

Never run two copies with the same `CHANNEL` at once, because both would fetch the same fires.

**Two phases.** The time-dependent channels cut each fire's days from `catalog/fire_extents.csv`, which the `firms`
copy writes when it finishes. A phase-2 copy started too early stops within a minute, saying how many fires are
still waiting; start it again later.

| phase | channels | when |
|---|---|---|
| 1 | `firms` (start it first), `history`, `perimeters`, `ics209`, `suppression`, `tessera`, `terrain`, `landfire`, `lulc`, `landcover`, `osm` | any time, all at once |
| 2 | `gridmet`, `goes`, `hrrr`, `hrrr_next`, `gfs`, `vegetation`, `viirs` | once `firms` is finished for every fire |
| any | `links` (rebuilds every fire's `LINKS.md` / `links.json`), `status` (progress per channel) | whenever you want a snapshot |

`goes`, `hrrr`, `hrrr_next`, `osm` and `viirs` take several 12-hour sessions; the others usually take one. Run a
copy again (a new version) until its channel is done: fires already in the repo's manifests are skipped, so each
session picks up where the last one stopped.

**Running copies on Kaggle**
1. *Copy & Edit* this notebook once per channel, and set `CHANNEL` (and the same `HF_REPO`) in each copy.
2. In each copy: Settings → Internet **on**, and under Add-ons → Secrets switch on the secrets its channel needs:

   | secret | needed by |
   |---|---|
   | `HF_TOKEN_ALL`: a Hugging Face token with write access | every channel (the label is set in `SECRET_LABELS`) |
   | `FIRMS_MAP_KEY`: free at https://firms.modaps.eosdis.nasa.gov/api/map_key/ | `firms` |
   | `EARTHDATA_TOKEN`: free at https://urs.earthdata.nasa.gov (Generate Token, valid 60 days) | `viirs`, `vegetation`, `landcover` |
   | `GEE_SERVICE_ACCOUNT_KEY` (JSON key of a Google Cloud service account with Earth Engine access), `GEE_PROJECT` (its project id; register at https://code.earthengine.google.com/register) | `gfs` |

3. *Save Version* → *Save & Run All (Commit)*: the copy runs in the background, in its own session, and you can
   start the next copy. Kaggle limits how many sessions an account runs at once; the rest wait in its queue.

In [ ]:
# ---- settings: CHANNEL is the one thing that differs between copies
CHANNEL = "firms"       # history perimeters firms gridmet ics209 suppression tessera terrain landfire lulc landcover
                        # vegetation goes hrrr hrrr_next gfs osm viirs, or "links" / "status"
HF_REPO = "your-name/wildfire-raw-2014-2025"   # target dataset repo: the same in every copy
PRIVATE = True                                 # only used when the repo is created
ARCHIVE = "all"                                # all | geomac | nifs | both: which catalog fires to fetch
MAX_HOURS = 11.5                               # whole session, under Kaggle's 12 h limit
BATCH_GB = 2.0                                 # upload every ~2 GB (or 100 fires)
PARTIAL_WINDOWS = False                        # phase 2: True runs although firms is not finished for every fire
                                               # (those fires are fetched again once their windows are final)

# Kaggle secret label (Add-ons > Secrets) for each value the script reads: left = what the script uses,
# right = the secret's label in Kaggle. Change the right side if your secrets are named differently.
SECRET_LABELS = {"HF_TOKEN": "HF_TOKEN_ALL", "FIRMS_MAP_KEY": "FIRMS_MAP_KEY", "EARTHDATA_TOKEN": "EARTHDATA_TOKEN",
                 "GEE_SERVICE_ACCOUNT_KEY": "GEE_SERVICE_ACCOUNT_KEY", "GEE_PROJECT": "GEE_PROJECT"}
BRANCH = "data_grab_satfire"                   # WildfireRepo1 branch the code comes from
WORK = f"/tmp/raw/{CHANNEL}"                   # scratch of this copy alone (not /kaggle/working)

In [ ]:
# ---- which channel is which
PHASE_1 = ["firms", "history", "perimeters", "ics209", "suppression", "tessera", "terrain", "landfire", "lulc",
           "landcover", "osm"]
PHASE_2 = ["gridmet", "goes", "hrrr", "hrrr_next", "gfs", "vegetation", "viirs"]   # read catalog/fire_extents.csv
NEEDS = {"firms": ["FIRMS_MAP_KEY"], "viirs": ["EARTHDATA_TOKEN"], "vegetation": ["EARTHDATA_TOKEN"],
         "landcover": ["EARTHDATA_TOKEN"], "gfs": ["GEE_SERVICE_ACCOUNT_KEY", "GEE_PROJECT"]}
assert CHANNEL in PHASE_1 + PHASE_2 + ["links", "status"], f"unknown CHANNEL {CHANNEL!r}"
print(f"this copy: {CHANNEL}" + (" (phase 2: waits for firms)" if CHANNEL in PHASE_2 else ""))

In [ ]:
# ---- the project code, BRANCH (settings cell)
!git clone -q --depth 1 -b {BRANCH} https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD)
!cd WildfireRepo1 && git log -1 --format='%h %s'

In [ ]:
# ---- packages. netCDF4 is upgraded on purpose: Kaggle's preinstalled build failed to open remote data,
# the PyPI wheels read OPeNDAP, HTTPS and in-memory files (tested)
!pip install -q -U netCDF4 pyogrio pygrib h5py pyhdf earthengine-api pyarrow "huggingface_hub>=0.24"
import shutil
if CHANNEL == "osm" and not shutil.which("osmium"):
    !apt-get -qq update && apt-get -qq install -y osmium-tool aria2 > /dev/null

In [ ]:
# ---- secrets -> the environment variables the script reads (only this channel's; values never printed)
import os
_wanted = ["HF_TOKEN"] + NEEDS.get(CHANNEL, [])
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _env in _wanted:
        try:
            os.environ[_env] = _s.get_secret(SECRET_LABELS[_env])
        except Exception as _e:
            print(f"Kaggle secret {SECRET_LABELS[_env]!r} not readable ({type(_e).__name__}): add it under Add-ons > "
                  f"Secrets with exactly this label and switch it on for this copy (or change SECRET_LABELS)")
except ImportError:
    pass                                       # not on Kaggle: use the existing environment
_missing = [e for e in _wanted if not os.environ.get(e)]
print({e: ("set" if os.environ.get(e) else "MISSING") for e in _wanted})
assert not _missing, f"{CHANNEL} needs {', '.join(_missing)}: switch the secret on for this copy, then rerun this cell"

In [ ]:
import subprocess, sys

REPO_DIR = "WildfireRepo1"                  # the code: package fah/, run as python -m fah from here

def _run(args):
    p = subprocess.Popen([sys.executable, "-u", "-m", "fah", *args], cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def _opts(opts):
    out = []
    for k, v in opts.items():
        flag = "--" + k.replace("_", "-")
        if v is True:
            out.append(flag)
        elif isinstance(v, (list, tuple)):
            out += [flag, *map(str, v)]
        elif v not in (None, False):
            out += [flag, str(v)]
    return out

def grab(channel, **opts):
    """One channel for every fire (resumes from the repo's manifests). opts: ids=[...], limit=N, dry_run=True, max_hours=..."""
    args = ["--channel", channel, "--hf-repo", HF_REPO, "--work", WORK, "--archive", opts.pop("archive", ARCHIVE),
            "--batch-gb", str(opts.pop("batch_gb", BATCH_GB))] + (["--private"] if PRIVATE else [])
    args += ["--partial-windows"] if PARTIAL_WINDOWS and channel in PHASE_2 else []
    return _run(args + _opts(opts))

def extents(**opts):
    """catalog/fire_extents.csv from the firms files: each fire's window, ignition point and region."""
    return _run(["--extents", "--hf-repo", HF_REPO, "--work", WORK, "--archive", opts.pop("archive", ARCHIVE)] + _opts(opts))

def links(**opts):
    """Rebuild fires/<id>/LINKS.md + links.json and catalog/raw_index.csv from the manifests."""
    return _run(["--links", "--hf-repo", HF_REPO, "--work", WORK, "--archive", opts.pop("archive", ARCHIVE)] + _opts(opts))

def status():
    return _run(["--status", "--hf-repo", HF_REPO])

In [ ]:
# ---- stop here unless the token can write to HF_REPO and the fire list is there (prints exactly what to fix)
assert _run(["--check", "--hf-repo", HF_REPO]) == 0, "fix the setup above, then rerun from the secrets cell"
from huggingface_hub import HfApi
assert HfApi().file_exists(HF_REPO, "catalog/fires.csv", repo_type="dataset"), \
    "no catalog/fires.csv in HF_REPO yet: run kaggle_catalog.ipynb on this repo first"

Optional check first: this channel for one fire, nothing uploaded. Camp Fire 2018 is `F2003403265`.

In [ ]:
# grab(CHANNEL, ids=["F2003403265"], dry_run=True)
# !find {WORK}/batch -type f | sort | head -40

## Run this copy's channel

In [ ]:
EXTENTS_RESERVE_H = 0.5                    # firms: time kept to write catalog/fire_extents.csv at the end
if CHANNEL == "links":
    rc = links()
elif CHANNEL == "status":
    rc = status()
else:
    hours = MAX_HOURS - (EXTENTS_RESERVE_H if CHANNEL == "firms" else 0)
    print(f"========== {CHANNEL} (up to {hours:.1f} h) ==========")
    rc = grab(CHANNEL, max_hours=round(hours, 2))
    if CHANNEL == "firms" and rc == 0:
        # every session: rows of fires whose firms is not finished say so (firms_current), and phase-2 copies wait
        print("\n========== fire extents: window, ignition, region (from firms) ==========")
        rc = extents()

## Progress of every channel

In [ ]:
if CHANNEL != "status":
    status()
WHY = {1: f"{CHANNEL} could not run here: see the message above (a missing secret or tool, or for a phase-2 "
          "channel, fires whose firms is not finished yet: run this copy again later, or set PARTIAL_WINDOWS)",
       3: "setup error (token or repo name): fix it, then run this copy again",
       4: f"{CHANNEL} failed for many fires in a row (see CHANNEL STOPPED above); its fires are retried next run"}
assert rc == 0, WHY.get(rc, f"exit code {rc}")